In [1]:
import pandas as pd

In [2]:
dfs_einstein = pd.read_excel("../data/EINSTEIN/20230724_Dados_Epidemiologicos_Semanais_ITpS_SE29.xlsx", sheet_name=None)
dfs_einstein = [df.assign(sheetName=sheet) for sheet, df in dfs_einstein.items() if not sheet.startswith("EXAMES")]
df_einstein = pd.concat(dfs_einstein, ignore_index=True)

## Listar todos os DETALHE_EXAME por tipo de EXAME

In [3]:
# pandas set limit of column size
pd.set_option('display.max_colwidth', -1)

/home/jaolima/anaconda3/envs/diag/lib/python3.7/site-packages/ipykernel_launcher.py:2: FutureWarning: Passing a negative integer is deprecated in version 1.0 and will not be supported in future version. Instead, use None to not limit the column width.
  


In [4]:
(
    df_einstein.fillna("N/A")
    .groupby("EXAME")
    .agg({"DETALHE_EXAME": "unique"})
    .assign(DETALHE_EXAME=lambda x: x["DETALHE_EXAME"].apply(lambda x: ", ".join(x)))
)

,DETALHE_EXAME
EXAME,
24 HRS COMPANHIA AÉREA - PCR COVID19,N/A
EXCLUSIVO EMPRESAS PCR COVID-19,N/A
HMSC - TESTE MOLECULAR ISOTÉRMICO COVID-,N/A
OPERAÇÃO AEROPORTO ANTÍGENO COVID-19,N/A
OPERAÇÃO AEROPORTO PCR COVID-19,N/A
PAINEL MOLECULAR PARA PNEUMONIA,"INFLUENZA A., INFLUENZA B., VIRUS SINCICIAL RESPIRATÓRIO."
PCR COVID19 EXPRESS,N/A
PCR EM TEMPO REAL PARA DETECÇÃO DE C,N/A
PCR EM TEMPO REAL PARA O VÍRUS MONKEYPOX,PCR MONKEYPOX


## Exames com ZZ na frente 

In [14]:
(
    df_einstein
    .query("EXAME.str.startswith('ZZ')", engine='python')
    .sample(10)
    .head()
)

,ACCESSION,SEXO,IDADE,EXAME,DH_COLETA,MUNICÍPIO,ESTADO,PATOGENO,RESULTADO,sheetName,DETALHE_EXAME
515542,2022362006964,FEMININO,74,ZZPAINEL MOLECULAR PARA PNEUMONIA,2022-12-28,SÃO PAULO,SÃO PAULO,INFLUENZA,NÃO DETECTADO,INFLUENZA,INFLUENZA B.
686554,2022363007676,MASCULINO,56,ZZPAINEL MOLECULAR PARA PNEUMONIA,2022-12-29,SÃO PAULO,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,NÃO DETECTADO,VSR,VIRUS SINCICIAL RESPIRATÓRIO.
518724,2022350008516,FEMININO,65,ZZPAINEL MOLECULAR PARA PNEUMONIA,2022-12-16,SÃO PAULO,SÃO PAULO,INFLUENZA,NÃO DETECTADO,INFLUENZA,INFLUENZA A.
219082,2022106008365,MASCULINO,40,ZZOPERAÇÃO AEROPORTO ANTÍGENO COVID-19,2022-04-16,GUARULHOS,SÃO PAULO,COVID,Não Detectado,COVID-19,NaN
297154,2022034008281,MASCULINO,43,"ZZTESTE MOLECULAR COVID-19, AMPLIFICAÇ",2022-02-03,SÃO PAULO,SÃO PAULO,COVID,Detectado,COVID-19,NaN


In [8]:
(
    df_einstein
    .query("EXAME.str.startswith('ZZ')", engine='python')
    .groupby('EXAME')
    .size()
)

EXAME
ZZOPERAÇÃO AEROPORTO ANTÍGENO COVID-19    33 
ZZOPERAÇÃO AEROPORTO PCR COVID-19         23 
ZZPAINEL MOLECULAR PARA PNEUMONIA         108
ZZPCR PAINEL DE PATOGENOS RESPIRATORIO    2  
ZZTESTE MOLECULAR COVID-19, AMPLIFICAÇ    12 
ZZTESTE RÁPIDO-ANTÍGENO COVID-19 (SARS    14 
dtype: int64

## Contagem de patógenos por amostra+exame (ACCESSION + EXAME)

In [5]:
df_contagem_patogenos_por_amostra = (
    df_einstein
    .assign(count=1)
    .groupby(['ACCESSION', 'EXAME'])
    .agg({'count': 'sum'})
    .sort_values(by='count', ascending=False)
)

### 5 Patógenos por amostra

In [6]:
df_contagem_patogenos_por_amostra.reset_index().query("count == 5").shape

(40364, 3)

In [7]:
df_contagem_patogenos_por_amostra.reset_index().query("count == 5")['EXAME'].unique()

array(['PCR PAINEL DE PATOGENOS RESPIRATORIO'], dtype=object)

In [8]:
df_einstein.query("ACCESSION == 2022143010059")

,ACCESSION,SEXO,IDADE,EXAME,DH_COLETA,MUNICÍPIO,ESTADO,PATOGENO,RESULTADO,sheetName,DETALHE_EXAME
174071,2022143010059,FEMININO,10,PCR PAINEL DE PATOGENOS RESPIRATORIO,2022-05-23,SÃO PAULO,SÃO PAULO,COVID,Não Detectado,COVID-19,NaN
615634,2022143010059,FEMININO,9,PCR PAINEL DE PATOGENOS RESPIRATORIO,2022-05-23,SÃO PAULO,SÃO PAULO,INFLUENZA,DETECTADO,INFLUENZA,INFLUENZA A
615699,2022143010059,FEMININO,9,PCR PAINEL DE PATOGENOS RESPIRATORIO,2022-05-23,SÃO PAULO,SÃO PAULO,INFLUENZA,NÃO DETECTADO,INFLUENZA,INF A H1N1 2009
615903,2022143010059,FEMININO,9,PCR PAINEL DE PATOGENOS RESPIRATORIO,2022-05-23,SÃO PAULO,SÃO PAULO,INFLUENZA,NÃO DETECTADO,INFLUENZA,INFLUENZA B
705541,2022143010059,FEMININO,9,PCR PAINEL DE PATOGENOS RESPIRATORIO,2022-05-23,SÃO PAULO,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,NÃO DETECTADO,VSR,VÍRUS SINCICIAL RESPIRATÓRIO


### 4 Patógenos

In [9]:
df_contagem_patogenos_por_amostra.reset_index().query("count == 4").shape

(8, 3)

In [10]:
df_contagem_patogenos_por_amostra.reset_index().query("count == 4")

,ACCESSION,EXAME,count
40364,2023205006474,PCR PAINEL DE PATOGENOS RESPIRATORIO,4
40365,2022065004508,PCR PAINEL DE PATOGENOS RESPIRATORIO,4
40366,2022034004413,PCR PAINEL DE PATOGENOS RESPIRATORIO,4
40367,2022071001463,PCR PAINEL DE PATOGENOS RESPIRATORIO,4
40368,2022038008026,PCR PAINEL DE PATOGENOS RESPIRATORIO,4
40369,2022193011698,PCR PAINEL DE PATOGENOS RESPIRATORIO,4
40370,2022034007621,PCR PAINEL DE PATOGENOS RESPIRATORIO,4
40371,2023151011001,PCR PAINEL DE PATOGENOS RESPIRATORIO,4


In [11]:
df_contagem_patogenos_por_amostra.reset_index().query("count == 4")['EXAME'].unique()

array(['PCR PAINEL DE PATOGENOS RESPIRATORIO'], dtype=object)

In [12]:
df_einstein.query("ACCESSION == 2023205006474")

,ACCESSION,SEXO,IDADE,EXAME,DH_COLETA,MUNICÍPIO,ESTADO,PATOGENO,RESULTADO,sheetName,DETALHE_EXAME
6,2023205006474,FEMININO,78,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,COVID,Não Detectado,COVID-19,NaN
427292,2023205006474,FEMININO,77,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,INFLUENZA,NÃO DETECTADO,INFLUENZA,INFLUENZA B
427305,2023205006474,FEMININO,77,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,INFLUENZA,NÃO DETECTADO,INFLUENZA,INFLUENZA A
672435,2023205006474,FEMININO,77,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,NÃO DETECTADO,VSR,VÍRUS SINCICIAL RESPIRATÓRIO


### 3 Patógenos

In [13]:
df_contagem_patogenos_por_amostra.reset_index().query("count == 3").shape

(1678, 3)

In [14]:
df_contagem_patogenos_por_amostra.reset_index().query("count == 3")

,ACCESSION,EXAME,count
40372,2023084011964,PAINEL MOLECULAR PARA PNEUMONIA,3
40373,2022132009354,PAINEL MOLECULAR PARA PNEUMONIA,3
40374,2023174008730,PAINEL MOLECULAR PARA PNEUMONIA,3
40375,2023136007897,PAINEL MOLECULAR PARA PNEUMONIA,3
40376,2023088010429,"PCR MULTIPLEX ZIKA, DENGUE E CHIKUNG",3
...,...,...,...
42045,2023042009217,PAINEL MOLECULAR PARA PNEUMONIA,3
42046,2023185006092,PAINEL MOLECULAR PARA PNEUMONIA,3
42047,2022231013286,"PCR MULTIPLEX ZIKA, DENGUE E CHIKUNG",3
42048,2023140009357,PAINEL MOLECULAR PARA PNEUMONIA,3


In [15]:
df_contagem_patogenos_por_amostra.reset_index().query("count == 3")['EXAME'].unique()

array(['PAINEL MOLECULAR PARA PNEUMONIA',
       'PCR MULTIPLEX ZIKA, DENGUE E CHIKUNG',
       'PCR PARA INFLUENZA A/B E VRS',
       'ZZPAINEL MOLECULAR PARA PNEUMONIA',
       'PCR PAINEL DE PATOGENOS RESPIRATORIO'], dtype=object)

Verificando um exemplo de cada ...

In [16]:
EXAMES_PATOGENOS = [
    "PAINEL MOLECULAR PARA PNEUMONIA",
    "PCR MULTIPLEX ZIKA, DENGUE E CHIKUNG",
    "PCR PARA INFLUENZA A/B E VRS",
    "ZZPAINEL MOLECULAR PARA PNEUMONIA",
    "PCR PAINEL DE PATOGENOS RESPIRATORIO",
]

In [17]:
for exame in EXAMES_PATOGENOS:
    print(
        df_contagem_patogenos_por_amostra.reset_index()
        .query("count == 3")
        .query("EXAME == @exame")
        .head(1)
    )

           ACCESSION                            EXAME  count
40372  2023084011964  PAINEL MOLECULAR PARA PNEUMONIA  3    
           ACCESSION                                 EXAME  count
40376  2023088010429  PCR MULTIPLEX ZIKA, DENGUE E CHIKUNG  3    
           ACCESSION                         EXAME  count
40378  2022265001863  PCR PARA INFLUENZA A/B E VRS  3    
           ACCESSION                              EXAME  count
40382  2022353011277  ZZPAINEL MOLECULAR PARA PNEUMONIA  3    
           ACCESSION                                 EXAME  count
41200  2023086007696  PCR PAINEL DE PATOGENOS RESPIRATORIO  3    


In [18]:
(
    df_einstein.query(
        "ACCESSION in (2023084011964,2023088010429,2022265001863,2022353011277,2023086007696)"
    ).sort_values(by="ACCESSION")[["ACCESSION", "EXAME", "DETALHE_EXAME", "PATOGENO"]]
)

,ACCESSION,EXAME,DETALHE_EXAME,PATOGENO
569733,2022265001863,PCR PARA INFLUENZA A/B E VRS,INFLUENZA B,INFLUENZA
569982,2022265001863,PCR PARA INFLUENZA A/B E VRS,INFLUENZA A,INFLUENZA
695465,2022265001863,PCR PARA INFLUENZA A/B E VRS,VÍRUS SINCICIAL RESPIRATÓRIO,VIRUS SINCICIAL RESPIRATÓRIO
517771,2022353011277,ZZPAINEL MOLECULAR PARA PNEUMONIA,INFLUENZA A.,INFLUENZA
517865,2022353011277,ZZPAINEL MOLECULAR PARA PNEUMONIA,INFLUENZA B.,INFLUENZA
687198,2022353011277,ZZPAINEL MOLECULAR PARA PNEUMONIA,VIRUS SINCICIAL RESPIRATÓRIO.,VIRUS SINCICIAL RESPIRATÓRIO
489020,2023084011964,PAINEL MOLECULAR PARA PNEUMONIA,INFLUENZA B.,INFLUENZA
489115,2023084011964,PAINEL MOLECULAR PARA PNEUMONIA,INFLUENZA A.,INFLUENZA
682027,2023084011964,PAINEL MOLECULAR PARA PNEUMONIA,VIRUS SINCICIAL RESPIRATÓRIO.,VIRUS SINCICIAL RESPIRATÓRIO
24178,2023086007696,PCR PAINEL DE PATOGENOS RESPIRATORIO,NaN,COVID


### 2 Patógenos

In [19]:
df_contagem_patogenos_por_amostra.reset_index().query("count == 2").shape

(72871, 3)

In [20]:
df_contagem_patogenos_por_amostra.reset_index().query("count == 2")

,ACCESSION,EXAME,count
42050,2022262008628,PESQUISA RÁPIDA PARA INFLUENZA A E B,2
42051,2022270011888,PESQUISA RÁPIDA PARA INFLUENZA A E B,2
42052,2022270007224,PESQUISA RÁPIDA PARA INFLUENZA A E B,2
42053,2022277014359,PESQUISA RÁPIDA PARA INFLUENZA A E B,2
42054,2022262007468,TESTE RÁPIDO PARA DENGUE IGM E NS1,2
...,...,...,...
114916,2022003003356,PESQUISA RÁPIDA PARA INFLUENZA A E B,2
114917,2022022001796,PESQUISA RÁPIDA PARA INFLUENZA A E B,2
114918,2022010016942,PESQUISA RÁPIDA PARA INFLUENZA A E B,2
114919,2022011006604,PESQUISA RÁPIDA PARA INFLUENZA A E B,2


In [21]:
df_contagem_patogenos_por_amostra.reset_index().query("count == 2").query("EXAME.str.contains('COVID')", engine='python')

,ACCESSION,EXAME,count
105474,2022010009862,"TESTE MOLECULAR COVID-19, AMPLIFICAÇÃO I",2
109186,2022019005627,"TESTE MOLECULAR COVID-19, AMPLIFICAÇÃO I",2


In [22]:
df_contagem_patogenos_por_amostra.reset_index().query("count == 2")['EXAME'].unique()

array(['PESQUISA RÁPIDA PARA INFLUENZA A E B',
       'TESTE RÁPIDO PARA DENGUE IGM E NS1', 'SOROLOGIA PARA DENGUE',
       'PCR PAINEL DE PATOGENOS RESPIRATORIO',
       'TESTE MOLECULAR COVID-19, AMPLIFICAÇÃO I'], dtype=object)

Verificando um exemplo de cada ...

In [23]:
EXAMES_PATOGENOS = [
    "PESQUISA RÁPIDA PARA INFLUENZA A E B",
    "TESTE RÁPIDO PARA DENGUE IGM E NS1",
    "SOROLOGIA PARA DENGUE",
    "PCR PAINEL DE PATOGENOS RESPIRATORIO",
    "TESTE MOLECULAR COVID-19, AMPLIFICAÇÃO I",
]

In [24]:
for exame in EXAMES_PATOGENOS:
    print(
        df_contagem_patogenos_por_amostra.reset_index()
        .query("count == 2")
        .query("EXAME == @exame")
        .head(1)
    )

           ACCESSION                                 EXAME  count
42050  2022262008628  PESQUISA RÁPIDA PARA INFLUENZA A E B  2    
           ACCESSION                               EXAME  count
42054  2022262007468  TESTE RÁPIDO PARA DENGUE IGM E NS1  2    
           ACCESSION                  EXAME  count
42304  2022270002362  SOROLOGIA PARA DENGUE  2    
           ACCESSION                                 EXAME  count
46934  2022263001839  PCR PAINEL DE PATOGENOS RESPIRATORIO  2    
            ACCESSION                                     EXAME  count
105474  2022010009862  TESTE MOLECULAR COVID-19, AMPLIFICAÇÃO I  2    


In [25]:
(
    df_einstein.query(
        "ACCESSION in (2022262008628,2022262007468,2022270002362,2022263001839,2022010009862)"
    ).sort_values(by="ACCESSION")[["ACCESSION", "EXAME", "DETALHE_EXAME", "PATOGENO"]]
)

,ACCESSION,EXAME,DETALHE_EXAME,PATOGENO
351917,2022010009862,"TESTE MOLECULAR COVID-19, AMPLIFICAÇÃO I",NaN,COVID
352561,2022010009862,"TESTE MOLECULAR COVID-19, AMPLIFICAÇÃO I",NaN,COVID
413367,2022262007468,TESTE RÁPIDO PARA DENGUE IGG,"DENGUE IGG, TESTE RÁPIDO",DENGUE
413368,2022262007468,TESTE RÁPIDO PARA DENGUE IGM E NS1,"DENGUE IGM, TESTE RÁPIDO",DENGUE
413369,2022262007468,TESTE RÁPIDO PARA DENGUE IGM E NS1,"DENGUE NS1, TESTE RÁPIDO",DENGUE
573409,2022262008628,PESQUISA RÁPIDA PARA INFLUENZA A E B,"INFLUENZA A, TESTE RÁPIDO",INFLUENZA
573427,2022262008628,PESQUISA RÁPIDA PARA INFLUENZA A E B,"INFLUENZA B, TESTE RÁPIDO",INFLUENZA
81489,2022263001839,PCR PAINEL DE PATOGENOS RESPIRATORIO,NaN,COVID
573170,2022263001839,PCR PAINEL DE PATOGENOS RESPIRATORIO,INFLUENZA A,INFLUENZA
412996,2022270002362,SOROLOGIA PARA DENGUE,DENGUE IGM,DENGUE


## Listagem de patógenos por amostra 

In [38]:
df_listagens_por_amostra = (
    df_einstein
    .assign(count=1)
    .groupby(['ACCESSION', 'EXAME'])
    .agg({'DETALHE_EXAME': 'unique'})
)

In [39]:
df_listagens_por_amostra['DETALHE_EXAME_LEN'] = df_listagens_por_amostra['DETALHE_EXAME'].apply(lambda x: len(x))
df_listagens_por_amostra['DETALHE_EXAME'] = df_listagens_por_amostra['DETALHE_EXAME'].apply(lambda x: "; ".join(set([str(z) for z in x])))

In [40]:
df_listagens_por_amostra.query("DETALHE_EXAME_LEN >1")['DETALHE_EXAME'].unique().tolist()

['INFLUENZA  A; VÍRUS SINCICIAL RESPIRATÓRIO; INFLUENZA  B',
 'INFLUENZA B,  TESTE RÁPIDO; INFLUENZA A, TESTE RÁPIDO',
 'INFLUENZA A, TESTE RÁPIDO; INFLUENZA B,  TESTE RÁPIDO',
 'VÍRUS SINCICIAL RESPIRATÓRIO; INFLUENZA  B; INFLUENZA  A',
 'DENGUE NS1, TESTE RÁPIDO; DENGUE IGM, TESTE RÁPIDO',
 'VIRUS SINCICIAL RESPIRATÓRIO.; INFLUENZA B.; INFLUENZA A.',
 'VÍRUS SINCICIAL RESPIRATÓRIO; INF A H1N1 2009; INFLUENZA  A; nan; INFLUENZA  B',
 'DENGUE IGG; DENGUE IGM',
 'VÍRUS DENGUE:; VÍRUS ZIKA:; VÍRUS CHIKUNGUNYA:',
 'VÍRUS SINCICIAL RESPIRATÓRIO; INFLUENZA  B; INF A H1N1 2009; INFLUENZA  A',
 'INFLUENZA  A; VÍRUS SINCICIAL RESPIRATÓRIO; nan; INFLUENZA  B',
 'VÍRUS SINCICIAL RESPIRATÓRIO; INFLUENZA  B; nan; INFLUENZA  A',
 'INF A H1N1 2009; nan',
 'nan; INFLUENZA  A',
 'VÍRUS SINCICIAL RESPIRATÓRIO; nan',
 'INF A H1N1 2009; nan; INFLUENZA  A']

In [ ]:
df_listagens_por_amostra['DETALHE_EXAME']

ACCESSION      EXAME                               
2021365008664  OPERAÇÃO AEROPORTO PCR COVID-19         [nan]                                                  
2022001000003  PCR EM TEMPO REAL PARA DETECÇÃO DE C    [nan]                                                  
2022001000004  PCR EM TEMPO REAL PARA DETECÇÃO DE C    [nan]                                                  
2022001000005  PCR EM TEMPO REAL PARA DETECÇÃO DE C    [nan]                                                  
2022001000006  PCR EM TEMPO REAL PARA DETECÇÃO DE C    [nan]                                                  
                                                       ...                                                    
2023205008692  PESQUISA RÁPIDA PARA INFLUENZA A E B    [INFLUENZA B,  TESTE RÁPIDO, INFLUENZA A, TESTE RÁPIDO]
2023205008704  TESTE RÁPIDO-ANTÍGENO COVID-19 (SARS    [nan]                                                  
2023205008906  PESQUISA RÁPIDA PARA INFLUENZA A E B    [INFL

## Listagem de inconsistências por exame

In [41]:
df_einstein.columns

Index(['ACCESSION', 'SEXO', 'IDADE', 'EXAME', 'DH_COLETA', 'MUNICÍPIO',
       'ESTADO', 'PATOGENO', 'RESULTADO', 'sheetName', 'DETALHE_EXAME'],
      dtype='object')

In [42]:
["SEXO", "IDADE", "DH_COLETA", "MUNICÍPIO", "ESTADO"]

['SEXO', 'IDADE', 'DH_COLETA', 'MUNICÍPIO', 'ESTADO']

In [43]:
df_listagens_por_amostra_outros = (
    df_einstein
    .assign(count=1)
    .groupby(['ACCESSION', 'EXAME'])
    .agg({'SEXO': 'unique', 'IDADE': 'unique', 'MUNICÍPIO': 'unique', 'ESTADO': 'unique', 'count': 'sum'})
)

In [65]:
df_listagens_por_amostra_outros['SOMA_DOS_LEN'] = (
    df_listagens_por_amostra_outros['SEXO'].apply(len) +
    df_listagens_por_amostra_outros['IDADE'].apply(len) +
    df_listagens_por_amostra_outros['MUNICÍPIO'].apply(len) +
    df_listagens_por_amostra_outros['ESTADO'].apply(len)
)

df_listagens_por_amostra_outros['LEN_IDADE'] = (
    df_listagens_por_amostra_outros['IDADE'].apply(len)
)

df_listagens_por_amostra_outros['DIFF_IDADE'] = (
    df_listagens_por_amostra_outros['IDADE'].apply( lambda x: max(x) - min(x) )
)

In [66]:
df_listagens_por_amostra_outros.query("SOMA_DOS_LEN < 4")

,,SEXO,IDADE,MUNICÍPIO,ESTADO,count,SOMA_DOS_LEN,LEN_SEXO,LEN_IDADE,DIFF_IDADE
ACCESSION,EXAME,,,,,,,,,


In [79]:
df_listagens_por_amostra_outros.query("SOMA_DOS_LEN == 5")

,,SEXO,IDADE,MUNICÍPIO,ESTADO,count,SOMA_DOS_LEN,LEN_SEXO,LEN_IDADE,DIFF_IDADE
ACCESSION,EXAME,,,,,,,,,
2022003002912,PCR PAINEL DE PATOGENOS RESPIRATORIO,[MASCULINO],"[8, 7]",[SÃO PAULO],[SÃO PAULO],5,5,1,2,1
2022003002914,PCR PAINEL DE PATOGENOS RESPIRATORIO,[FEMININO],"[6, 5]",[SÃO PAULO],[SÃO PAULO],5,5,1,2,1
2022003009151,PCR PAINEL DE PATOGENOS RESPIRATORIO,[FEMININO],"[54, 53]",[SÃO PAULO],[SÃO PAULO],5,5,1,2,1
2022003011248,PCR PAINEL DE PATOGENOS RESPIRATORIO,[MASCULINO],"[46, 45]",[SÃO PAULO],[SÃO PAULO],5,5,1,2,1
2022003014148,PCR PAINEL DE PATOGENOS RESPIRATORIO,[FEMININO],"[91, 90]",[SÃO PAULO],[SÃO PAULO],5,5,1,2,1
...,...,...,...,...,...,...,...,...,...,...
2023204004801,PCR PAINEL DE PATOGENOS RESPIRATORIO,[MASCULINO],"[1, 0]",[SÃO PAULO],[SÃO PAULO],5,5,1,2,1
2023204005297,PCR PAINEL DE PATOGENOS RESPIRATORIO,[FEMININO],"[72, 71]",[SÃO PAULO],[SÃO PAULO],5,5,1,2,1
2023205006395,PCR PAINEL DE PATOGENOS RESPIRATORIO,[MASCULINO],"[46, 45]",[SÃO PAULO],[SÃO PAULO],5,5,1,2,1


In [76]:
df_listagens_por_amostra_outros.query("SOMA_DOS_LEN > 4 and LEN_IDADE < 2")

,,SEXO,IDADE,MUNICÍPIO,ESTADO,count,SOMA_DOS_LEN,LEN_SEXO,LEN_IDADE,DIFF_IDADE
ACCESSION,EXAME,,,,,,,,,


In [77]:
df_listagens_por_amostra_outros.query("SOMA_DOS_LEN > 5")

,,SEXO,IDADE,MUNICÍPIO,ESTADO,count,SOMA_DOS_LEN,LEN_SEXO,LEN_IDADE,DIFF_IDADE
ACCESSION,EXAME,,,,,,,,,
2022170003387,PCR PAINEL DE PATOGENOS RESPIRATORIO,"[MASCULINO, FEMININO]","[11, 10]",[SÃO PAULO],[SÃO PAULO],5,6,2,2,1


Todas as inconsistências possuem multiplicidade de idade

In [71]:
df_listagens_por_amostra_outros["DIFF_IDADE"].unique()

array([  0,   1, 119])

In [73]:
df_listagens_por_amostra_outros.query("DIFF_IDADE > 2")

,,SEXO,IDADE,MUNICÍPIO,ESTADO,count,SOMA_DOS_LEN,LEN_SEXO,LEN_IDADE,DIFF_IDADE
ACCESSION,EXAME,,,,,,,,,
2022021009380,PCR PAINEL DE PATOGENOS RESPIRATORIO,[FEMININO],"[123, 4]",[SÃO PAULO],[SÃO PAULO],5,5,1,2,119


## Checking Duplicates

In [ ]:
df_einstein_DUP = df_einstein.copy()

In [ ]:
df_einstein_DUP.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714617 entries, 0 to 714616
Data columns (total 11 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   ACCESSION      714617 non-null  int64         
 1   SEXO           714617 non-null  object        
 2   IDADE          714617 non-null  int64         
 3   EXAME          714617 non-null  object        
 4   DH_COLETA      714617 non-null  datetime64[ns]
 5   MUNICÍPIO      714617 non-null  object        
 6   ESTADO         714617 non-null  object        
 7   PATOGENO       714617 non-null  object        
 8   RESULTADO      714617 non-null  object        
 9   sheetName      714617 non-null  object        
 10  DETALHE_EXAME  321850 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(8)
memory usage: 60.0+ MB


In [ ]:
ID_COLUMNS = ['ACCESSION', 'EXAME', 'DETALHE_EXAME']
df_einstein_DUP['duplicated'] = df_einstein_DUP.duplicated(subset=ID_COLUMNS)

In [ ]:
df_einstein_DUP.groupby('sheetName').agg({'duplicated': ['sum', 'mean', 'count']})

duplicated                  
                     sum      mean   count
sheetName                                 
COVID-19       2          0.000005  392767
DENGUE         0          0.000000  34483 
INFLUENZA      0          0.000000  244899
VARIOLA_SIMIA  0          0.000000  283   
VSR            0          0.000000  42185

In [ ]:
df_einstein_DUP.query('duplicated')

,ACCESSION,SEXO,IDADE,EXAME,DH_COLETA,MUNICÍPIO,ESTADO,PATOGENO,RESULTADO,sheetName,DETALHE_EXAME,duplicated
328541,2022019005627,MASCULINO,20,"TESTE MOLECULAR COVID-19, AMPLIFICAÇÃO I",2022-01-19,SÃO PAULO,SÃO PAULO,COVID,Não Detectado,COVID-19,NaN,True
352561,2022010009862,MASCULINO,63,"TESTE MOLECULAR COVID-19, AMPLIFICAÇÃO I",2022-01-10,SÃO PAULO,SÃO PAULO,COVID,Detectado,COVID-19,NaN,True


In [ ]:
df_einstein_DUP.query("ACCESSION == 2023204005355")

,ACCESSION,SEXO,IDADE,EXAME,DH_COLETA,MUNICÍPIO,ESTADO,PATOGENO,RESULTADO,sheetName,DETALHE_EXAME,duplicated
0,2023204005355,FEMININO,71,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,COVID,Não Detectado,COVID-19,NaN,False
7,2023204005355,FEMININO,71,TESTE RÁPIDO-ANTÍGENO COVID-19 (SARS,2023-07-24,SÃO PAULO,SÃO PAULO,COVID,Não Detectado,COVID-19,NaN,False
427260,2023204005355,FEMININO,71,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,INFLUENZA,NÃO DETECTADO,INFLUENZA,INFLUENZA A,False
427272,2023204005355,FEMININO,71,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,INFLUENZA,DETECTADO,INFLUENZA,INFLUENZA B,False
427283,2023204005355,FEMININO,71,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,INFLUENZA,NÃO DETECTADO,INFLUENZA,INF A H1N1 2009,False
672432,2023204005355,FEMININO,71,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,NÃO DETECTADO,VSR,VÍRUS SINCICIAL RESPIRATÓRIO,False


In [ ]:
df_einstein_DUP['PATOGENO'].unique()

array(['COVID', 'DENGUE', 'INFLUENZA', 'VARIOLA SIMIA',
       'VIRUS SINCICIAL RESPIRATÓRIO'], dtype=object)

In [ ]:
df_einstein_DUP.query("duplicated == True").query("PATOGENO == 'VIRUS SINCICIAL RESPIRATÓRIO'")

,ACCESSION,SEXO,IDADE,EXAME,DH_COLETA,MUNICÍPIO,ESTADO,PATOGENO,RESULTADO,sheetName,DETALHE_EXAME,duplicated
672432,2023204005355,FEMININO,71,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,NÃO DETECTADO,VSR,VÍRUS SINCICIAL RESPIRATÓRIO,True
672433,2023205005744,FEMININO,57,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,ALPHAVILLE,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,NÃO DETECTADO,VSR,VÍRUS SINCICIAL RESPIRATÓRIO,True
672434,2023205006395,MASCULINO,45,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,NÃO DETECTADO,VSR,VÍRUS SINCICIAL RESPIRATÓRIO,True
672435,2023205006474,FEMININO,77,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,NÃO DETECTADO,VSR,VÍRUS SINCICIAL RESPIRATÓRIO,True
672436,2023205006494,MASCULINO,61,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,NÃO DETECTADO,VSR,VÍRUS SINCICIAL RESPIRATÓRIO,True
...,...,...,...,...,...,...,...,...,...,...,...,...
714612,2022001000793,FEMININO,1,PCR PARA INFLUENZA A/B E VRS,2022-01-01,SÃO PAULO,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,NÃO DETECTADO,VSR,VÍRUS SINCICIAL RESPIRATÓRIO,True
714613,2022001001981,FEMININO,1,PCR PARA INFLUENZA A/B E VRS,2022-01-01,SÃO PAULO,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,DETECTADO,VSR,VÍRUS SINCICIAL RESPIRATÓRIO,True
714614,2022001003087,FEMININO,75,PCR PARA INFLUENZA A/B E VRS,2022-01-01,SÃO PAULO,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,NÃO DETECTADO,VSR,VÍRUS SINCICIAL RESPIRATÓRIO,True
714615,2022001003240,MASCULINO,64,PCR PARA INFLUENZA A/B E VRS,2022-01-01,SÃO PAULO,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,DETECTADO,VSR,VÍRUS SINCICIAL RESPIRATÓRIO,True


## Fixing data

In [ ]:
import hashlib

In [ ]:
PATHOGEN_COLUMNS = [
    "COVID",
    "DENGUE",
    "INFLUENZA A",
    "INFLUENZA B",
    "VARIOLA SIMIA",
    "VIRUS SINCICIAL RESPIRATÓRIO",
]

ID_COLUMNS = [
    # id columns
    "ACCESSION",
    "EXAME",
    "DETALHE_EXAME",
    "SEXO",
    "IDADE",
    "MUNICÍPIO",
    "ESTADO",
    "DH_COLETA",
]

In [ ]:
df_einstein.shape

(714617, 11)

In [ ]:
df_einstein_pivot = (
    df_einstein.assign(
        DETALHE_EXAME=lambda df: df["DETALHE_EXAME"].fillna("S/ DETALHE"),
    )
    .assign(
        RESULTADO=lambda df: df["RESULTADO"]
        .str.lower()
        .str.strip()
        .map(
            {
                "não detectado": "neg",
                "detectado": "pos",
            }
        )
    )
    .assign(
        # CORRIGINDO PATOGENO INFLUENZA
        PATOGENO=lambda df: df.apply(
            lambda row: "INFLUENZA B"
            if "INFLUENZA B" in row["DETALHE_EXAME"]
            else "INFLUENZA A"
            if "INF A" in row["DETALHE_EXAME"] or "INFLUENZA" in row["DETALHE_EXAME"]
            else row["PATOGENO"],
            axis=1,
        )
    )
    .pivot_table(
        index=ID_COLUMNS,
        columns="PATOGENO",
        values="RESULTADO",
        aggfunc="first",
        fill_value="NT"
    )
    .reset_index()
)

In [ ]:
df_einstein_pivot.shape

(714615, 14)

In [ ]:
df_einstein_pivot['EXAME'].unique().tolist()

['OPERAÇÃO AEROPORTO PCR COVID-19',
 'PCR EM TEMPO REAL PARA DETECÇÃO DE C',
 'TESTE RÁPIDO-ANTÍGENO COVID-19 (SARS COV',
 'PCR PARA INFLUENZA A/B E VRS',
 'PESQUISA RÁPIDA PARA INFLUENZA A E B',
 'ZZOPERAÇÃO AEROPORTO PCR COVID-19',
 'TESTE MOLECULAR COVID-19, AMPLIFICAÇÃO I',
 'TESTE RÁPIDO PARA DENGUE IGM E NS1',
 'PCR COVID19 EXPRESS',
 'TESTE RÁPIDO PARA DENGUE IGG',
 'PAINEL MOLECULAR PARA PNEUMONIA',
 '24 HRS COMPANHIA AÉREA - PCR COVID19',
 'TESTE RÁPIDO-ANTÍGENO COVID-19 (SARS',
 'PCR PAINEL DE PATOGENOS RESPIRATORIO',
 'SOROLOGIA PARA DENGUE',
 'TESTE MOLECULAR COVID-19, AMPLIFICAÇ',
 'HMSC - TESTE MOLECULAR ISOTÉRMICO COVID-',
 'PCR MULTIPLEX ZIKA, DENGUE E CHIKUNG',
 'ZZPCR PAINEL DE PATOGENOS RESPIRATORIO',
 'ZZTESTE MOLECULAR COVID-19, AMPLIFICAÇ',
 'PESQ RÁPIDA VÍRUS SINCICIAL RESPIRAT',
 'OPERAÇÃO AEROPORTO ANTÍGENO COVID-19',
 'ZZOPERAÇÃO AEROPORTO ANTÍGENO COVID-19',
 'EXCLUSIVO EMPRESAS PCR COVID-19',
 'ZZTESTE RÁPIDO-ANTÍGENO COVID-19 (SARS',
 'PCR EM TEMPO REAL PAR

In [ ]:
df_einstein_pivot.query("ACCESSION == 2023205006395")

PATOGENO,ACCESSION,EXAME,DETALHE_EXAME,SEXO,IDADE,MUNICÍPIO,ESTADO,DH_COLETA,COVID,DENGUE,INFLUENZA A,INFLUENZA B,VARIOLA SIMIA,VIRUS SINCICIAL RESPIRATÓRIO
714560,2023205006395,PCR PAINEL DE PATOGENOS RESPIRATORIO,INF A H1N1 2009,MASCULINO,45,SÃO PAULO,SÃO PAULO,2023-07-24,NT,NT,neg,NT,NT,NT
714561,2023205006395,PCR PAINEL DE PATOGENOS RESPIRATORIO,INFLUENZA A,MASCULINO,45,SÃO PAULO,SÃO PAULO,2023-07-24,NT,NT,neg,NT,NT,NT
714562,2023205006395,PCR PAINEL DE PATOGENOS RESPIRATORIO,INFLUENZA B,MASCULINO,45,SÃO PAULO,SÃO PAULO,2023-07-24,NT,NT,neg,NT,NT,NT
714563,2023205006395,PCR PAINEL DE PATOGENOS RESPIRATORIO,S/ DETALHE,MASCULINO,46,SÃO PAULO,SÃO PAULO,2023-07-24,neg,NT,NT,NT,NT,NT
714564,2023205006395,PCR PAINEL DE PATOGENOS RESPIRATORIO,VÍRUS SINCICIAL RESPIRATÓRIO,MASCULINO,45,SÃO PAULO,SÃO PAULO,2023-07-24,NT,NT,NT,NT,NT,neg


In [ ]:
["ACCESSION", "EXAME", "DETALHE_EXAME"]

hashlib.sha1(str(value).encode("utf-8")).hexdigest()

In [ ]:
df_einstein.query("ACCESSION == 2023205006395")

,ACCESSION,SEXO,IDADE,EXAME,DH_COLETA,MUNICÍPIO,ESTADO,PATOGENO,RESULTADO,sheetName,DETALHE_EXAME
1,2023205006395,MASCULINO,46,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,COVID,Não Detectado,COVID-19,NaN
427251,2023205006395,MASCULINO,45,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,INFLUENZA,NÃO DETECTADO,INFLUENZA,INF A H1N1 2009
427252,2023205006395,MASCULINO,45,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,INFLUENZA,NÃO DETECTADO,INFLUENZA,INFLUENZA A
427274,2023205006395,MASCULINO,45,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,INFLUENZA,NÃO DETECTADO,INFLUENZA,INFLUENZA B
672434,2023205006395,MASCULINO,45,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,NÃO DETECTADO,VSR,VÍRUS SINCICIAL RESPIRATÓRIO
